# Sorting and Binary Search

Notebook for **Chapter 4** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch04.html

Tip: change a number, predict the output, then run the cell again.

## 4.1 Why Sort, and How Fast Can Sorting Be?

## 4.2 The Three Simple Sorts: Insertion, Selection, Bubble

These are slow on big inputs, O(n2), but their ideas (a sorted prefix, swap into final place, early exit) reappear inside harder problems. Insertion sort matters most.

In [ ]:
def insertion_sort(a):
    a = a[:]                           # copy: never change the caller's list
    for i in range(1, len(a)):         # a[0..i-1] is already sorted
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:   # strict > keeps equal items in order (stable)
            a[j + 1] = a[j]            # shift the bigger item right
            j -= 1
        a[j + 1] = key                 # drop the key into the gap
    return a

assert insertion_sort([5, 2, 4, 6, 1, 3]) == [1, 2, 3, 4, 5, 6]
print(insertion_sort([5, 2, 4, 6, 1, 3]))

[1, 2, 3, 4, 5, 6]


In [ ]:
def selection_sort(a):
    a = a[:]
    for i in range(len(a) - 1):        # a[0..i-1] holds the i smallest, final
        smallest = i
        for j in range(i + 1, len(a)):
            if a[j] < a[smallest]:
                smallest = j
        a[i], a[smallest] = a[smallest], a[i]
    return a

assert selection_sort([64, 25, 12, 22, 11]) == [11, 12, 22, 25, 64]
print(selection_sort([64, 25, 12, 22, 11]))

[11, 12, 22, 25, 64]


In [ ]:
def bubble_sort(a):
    a = a[:]
    for end in range(len(a) - 1, 0, -1):   # a[end+1..] is already final
        swapped = False
        for j in range(end):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
        if not swapped:                    # a clean sweep means sorted: stop early
            break
    return a

assert bubble_sort([5, 1, 4, 2, 8]) == [1, 2, 4, 5, 8]
print(bubble_sort([5, 1, 4, 2, 8]))

[1, 2, 4, 5, 8]


In [ ]:
def selection_by_value(r):
    r = r[:]
    for i in range(len(r) - 1):
        m = min(range(i, len(r)), key=lambda j: r[j][0])   # first minimum
        r[i], r[m] = r[m], r[i]
    return r

def insertion_by_value(r):
    r = r[:]
    for i in range(1, len(r)):
        key, j = r[i], i - 1
        while j >= 0 and r[j][0] > key[0]:
            r[j + 1] = r[j]
            j -= 1
        r[j + 1] = key
    return r

data = [(2, "a"), (2, "b"), (1, "c")]
assert insertion_by_value(data) == sorted(data, key=lambda x: x[0])
assert selection_by_value(data) != sorted(data, key=lambda x: x[0])
print(selection_by_value(data))
print(insertion_by_value(data))

[(1, 'c'), (2, 'b'), (2, 'a')]
[(1, 'c'), (2, 'a'), (2, 'b')]


## 4.3 Merge Sort: Divide and Conquer

In [ ]:
def merge(left, right):
    """Merge two sorted lists in O(len(left) + len(right))."""
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:        # <= takes the LEFT item on ties: stable
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:]  # one side is empty; the other tail is sorted

def merge_sort(a):
    if len(a) <= 1:                    # 0 or 1 items are already sorted
        return a[:]
    mid = len(a) // 2
    return merge(merge_sort(a[:mid]), merge_sort(a[mid:]))

data = [38, 27, 43, 3, 9, 82, 10]
assert merge_sort(data) == sorted(data)
assert merge([1, 4, 7], [2, 3, 9, 10]) == [1, 2, 3, 4, 7, 9, 10]
print(merge_sort(data))

[3, 9, 10, 27, 38, 43, 82]


In [ ]:
def sort_and_count(a):
    """Return (sorted copy, number of inversions)."""
    if len(a) <= 1:
        return a[:], 0
    mid = len(a) // 2
    left, c1 = sort_and_count(a[:mid])
    right, c2 = sort_and_count(a[mid:])
    out, i, j, cross = [], 0, 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
            cross += len(left) - i     # every item still waiting in left is bigger
    return out + left[i:] + right[j:], c1 + c2 + cross

assert sort_and_count([2, 4, 1, 3, 5])[1] == 3
assert sort_and_count([2, 2])[1] == 0
assert sort_and_count(list(range(10, 0, -1)))[1] == 45
print(sort_and_count([2, 4, 1, 3, 5]))

([1, 2, 3, 4, 5], 3)


## 4.4 Quick Sort and Partition

In [ ]:
import random

def partition(a, lo, hi):
    """Partition a[lo..hi] around pivot a[hi]; return the pivot's final index."""
    pivot, i = a[hi], lo - 1           # a[lo..i] holds items <= pivot
    for j in range(lo, hi):            # j scans the unseen items
        if a[j] <= pivot:
            i += 1
            a[i], a[j] = a[j], a[i]    # grow the small zone by one
    a[i + 1], a[hi] = a[hi], a[i + 1]  # put the pivot between the zones
    return i + 1

def quick_sort(a, lo=0, hi=None):
    hi = len(a) - 1 if hi is None else hi
    if lo >= hi:                       # 0 or 1 items: done
        return
    r = random.randint(lo, hi)         # random pivot defeats nasty inputs
    a[r], a[hi] = a[hi], a[r]
    p = partition(a, lo, hi)
    quick_sort(a, lo, p - 1)
    quick_sort(a, p + 1, hi)

test = [8, 3, 7, 1, 6, 2, 5]
print(partition(test, 0, 6), test)
random.seed(1)
data = [8, 3, 7, 1, 6, 2, 5]
quick_sort(data)
assert data == [1, 2, 3, 5, 6, 7, 8]

3 [3, 1, 2, 5, 6, 7, 8]


In [ ]:
import random

def kth_largest(nums, k):
    a, target = nums[:], len(nums) - k     # target = index in sorted order
    lo, hi = 0, len(a) - 1
    while True:
        r = random.randint(lo, hi)
        a[r], a[hi] = a[hi], a[r]
        pivot, i = a[hi], lo - 1
        for j in range(lo, hi):
            if a[j] <= pivot:
                i += 1
                a[i], a[j] = a[j], a[i]
        a[i + 1], a[hi] = a[hi], a[i + 1]
        p = i + 1                          # pivot's final index
        if p == target:
            return a[p]
        if p < target:
            lo = p + 1                     # answer is on the right
        else:
            hi = p - 1                     # answer is on the left

random.seed(3)
assert kth_largest([3, 2, 1, 5, 6, 4], 3) == 4
assert kth_largest([7, 7, 7], 2) == 7 and kth_largest([9], 1) == 9
for k in range(1, 7):
    assert kth_largest([3, 2, 1, 5, 6, 4], k) == sorted([3, 2, 1, 5, 6, 4], reverse=True)[k - 1]
print(kth_largest([3, 2, 1, 5, 6, 4], 3))

4


In [ ]:
def sort_colors(a):
    a = a[:]
    lo, mid, hi = 0, 0, len(a) - 1
    while mid <= hi:
        if a[mid] == 0:
            a[lo], a[mid] = a[mid], a[lo]
            lo += 1
            mid += 1                   # the swapped-in item came from the 1s zone: safe
        elif a[mid] == 1:
            mid += 1
        else:
            a[mid], a[hi] = a[hi], a[mid]
            hi -= 1                    # do NOT advance mid: the new a[mid] is unchecked
    return a

assert sort_colors([2, 0, 2, 1, 1, 0]) == [0, 0, 1, 1, 2, 2]
assert sort_colors([]) == [] and sort_colors([1, 2, 0]) == [0, 1, 2]
print(sort_colors([2, 0, 2, 1, 1, 0]))

[0, 0, 1, 1, 2, 2]


## 4.5 Beating n log n, Stability and How Python Sorts

In [ ]:
def counting_sort(a, k):
    """Sort integers in 0..k in O(n + k) time and O(k) extra space."""
    counts = [0] * (k + 1)             # counts[v] = how many times v appears
    for x in a:
        counts[x] += 1
    out = []
    for value, c in enumerate(counts):
        out.extend([value] * c)
    return out

def radix_sort(a):
    """LSD radix sort for non-negative integers: one stable pass per digit."""
    a, place = a[:], 1                 # place 1 = ones digit, 10 = tens digit, ...
    while place <= max(a, default=0):
        buckets = [[] for _ in range(10)]
        for x in a:
            buckets[(x // place) % 10].append(x)   # append keeps input order: stable
        a = [x for b in buckets for x in b]
        print("place", place, a)
        place *= 10
    return a

assert counting_sort([3, 1, 3, 0, 2, 1], 3) == [0, 1, 1, 2, 3, 3]
data = [170, 45, 75, 90, 802, 24, 2, 66]
assert radix_sort(data) == sorted(data)

place 1 [170, 90, 802, 2, 24, 45, 75, 66]
place 10 [802, 2, 24, 45, 66, 170, 75, 90]
place 100 [2, 24, 45, 66, 75, 90, 170, 802]


In [ ]:
import random

calls = 0
class Counted:
    def __init__(self, v):
        self.v = v
    def __lt__(self, other):           # sort only ever uses <, so count it
        global calls
        calls += 1
        return self.v < other.v

def comparisons(values):
    global calls
    calls = 0
    sorted(Counted(v) for v in values)
    return calls

n = 1000
random.seed(2)
asc = comparisons(list(range(n)))
desc = comparisons(list(range(n, 0, -1)))
rnd = comparisons(random.sample(range(n), n))
assert asc == n - 1
print("ascending", asc, "descending", desc, "random", rnd)

ascending 999 descending 999 random 8667


In [ ]:
words = ["pear", "fig", "banana", "kiwi", "apple", "Fig"]
print(sorted(words))                           # capitals sort before lowercase
print(sorted(words, key=str.lower))            # case-insensitive; ties keep input order
print(sorted(words, key=lambda w: (-len(w), w)))   # longest first, then alphabetical
nums = [3, 1, 2]
result = nums.sort()                           # in place: returns None, not the list
print(result, nums)

['Fig', 'apple', 'banana', 'fig', 'kiwi', 'pear']
['apple', 'banana', 'fig', 'Fig', 'kiwi', 'pear']
['banana', 'apple', 'kiwi', 'pear', 'Fig', 'fig']
None [1, 2, 3]


In [ ]:
staff = [("Ben", "Ops"), ("Neha", "Eng"), ("Raj", "Ops"), ("Tara", "Eng")]
step1 = sorted(staff, key=lambda s: s[0], reverse=True)   # secondary key first
step2 = sorted(step1, key=lambda s: s[1])                 # main key second, stable
assert step2 == [("Tara", "Eng"), ("Neha", "Eng"), ("Raj", "Ops"), ("Ben", "Ops")]
print(step1)
print(step2)

[('Tara', 'Eng'), ('Raj', 'Ops'), ('Neha', 'Eng'), ('Ben', 'Ops')]
[('Tara', 'Eng'), ('Neha', 'Eng'), ('Raj', 'Ops'), ('Ben', 'Ops')]


## 4.6 Sort-Then-Solve Patterns: Intervals and Meeting Rooms

In [ ]:
def can_attend(meetings):
    s = sorted(meetings)                   # lists sort by start first
    return all(cur[0] >= prev[1] for prev, cur in zip(s, s[1:]))

assert can_attend([[0, 30], [5, 10], [15, 20]]) is False
assert can_attend([[7, 10], [2, 4]]) is True
assert can_attend([[2, 4], [4, 6]]) is True and can_attend([]) is True
print(can_attend([[0, 30], [5, 10], [15, 20]]), can_attend([[7, 10], [2, 4]]))

False True


In [ ]:
def merge_intervals(intervals):
    result = []
    for start, end in sorted(intervals):
        if result and start <= result[-1][1]:         # overlaps the last merged interval
            result[-1][1] = max(result[-1][1], end)   # extend, never shrink
        else:
            result.append([start, end])
    return result

data = [[15, 18], [2, 6], [17, 20], [1, 3], [8, 10]]
assert merge_intervals(data) == [[1, 6], [8, 10], [15, 20]]
assert merge_intervals([[1, 10], [2, 3]]) == [[1, 10]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]] and merge_intervals([]) == []
print(merge_intervals(data))

[[1, 6], [8, 10], [15, 20]]


In [ ]:
def min_rooms(meetings):
    starts = sorted(m[0] for m in meetings)
    ends = sorted(m[1] for m in meetings)
    rooms = e = 0
    for s in starts:
        if s < ends[e]:        # earliest end is still in the future: need a new room
            rooms += 1
        else:                  # that meeting finished: reuse its room
            e += 1
    return rooms

assert min_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert min_rooms([[7, 10], [2, 4]]) == 1 and min_rooms([[1, 5], [5, 8]]) == 1
assert min_rooms([[1, 4], [2, 5], [3, 6]]) == 3 and min_rooms([]) == 0
print(min_rooms([[0, 30], [5, 10], [15, 20]]), min_rooms([[1, 4], [2, 5], [3, 6]]))

2 3


In [ ]:
from functools import cmp_to_key

def largest_number(nums):
    strs = [str(x) for x in nums]
    def compare(a, b):
        if a + b > b + a:
            return -1                      # a first
        if a + b < b + a:
            return 1                       # b first
        return 0
    strs.sort(key=cmp_to_key(compare))
    joined = "".join(strs)
    return "0" if joined[0] == "0" else joined

assert largest_number([3, 30, 34, 5, 9]) == "9534330"
assert largest_number([0, 0]) == "0" and largest_number([10, 2]) == "210"
print(largest_number([3, 30, 34, 5, 9]), largest_number([0, 0]))

9534330 0


## 4.7 Binary Search on a Sorted Array

In [ ]:
def binary_search(a, target):
    """Template A: exact match in a sorted list. Return the index or -1."""
    lo, hi = 0, len(a) - 1             # live range a[lo..hi], BOTH ends included
    while lo <= hi:                    # <= because lo == hi is still one live item
        mid = (lo + hi) // 2
        if a[mid] == target:
            return mid
        if a[mid] < target:
            lo = mid + 1               # mid is checked and too small: step PAST it
        else:
            hi = mid - 1               # mid is checked and too big: step PAST it
    return -1

def first_true(n, ok):
    """Template B: smallest i in 0..n-1 with ok(i) True, or n if none.
    ok(i) must look like False, False, True, True (it flips once)."""
    lo, hi = 0, n                      # live range [lo, hi); hi = n means 'none'
    while lo < hi:                     # < because [lo, lo) is empty
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid                   # mid might be the answer: KEEP it
        else:
            lo = mid + 1               # mid is not the answer: step past it
    return lo

nums = [2, 5, 8, 12, 16, 23, 38, 45, 56, 72, 91]
assert binary_search(nums, 72) == 9 and binary_search(nums, 7) == -1
assert binary_search([], 3) == -1 and binary_search([4], 4) == 0
assert first_true(len(nums), lambda i: nums[i] >= 38) == 6
print(binary_search(nums, 72), binary_search(nums, 7), first_true(len(nums), lambda i: nums[i] >= 38))

9 -1 6


In [ ]:
import bisect

a = [1, 2, 2, 2, 2, 5, 7]
print(bisect.bisect_left(a, 2), bisect.bisect_right(a, 2))   # 1 and 5
print(bisect.bisect_left(a, 4), bisect.bisect_right(a, 4))   # both 5: 4 is absent
print(bisect.bisect_right(a, 2) - bisect.bisect_left(a, 2))  # how many 2s
b = [1, 3, 6]
bisect.insort(b, 4)                    # insert while keeping the list sorted
print(b)

1 5
5 5
4
[1, 3, 4, 6]


In [ ]:
import bisect

def search_range(a, target):
    first = bisect.bisect_left(a, target)               # lower bound
    if first == len(a) or a[first] != target:           # absent (check bounds first!)
        return [-1, -1]
    return [first, bisect.bisect_right(a, target) - 1]  # upper bound minus one

def count_in_range(a, low, high):
    if low > high:
        return 0
    return bisect.bisect_right(a, high) - bisect.bisect_left(a, low)

a = [5, 7, 7, 8, 8, 10]
assert search_range(a, 8) == [3, 4] and search_range(a, 6) == [-1, -1]
assert search_range([], 1) == [-1, -1] and search_range([2, 2, 2], 2) == [0, 2]
times = [1, 3, 3, 5, 8, 8, 8, 13]
assert count_in_range(times, 3, 8) == 6 and count_in_range(times, 9, 12) == 0
print(search_range(a, 8), search_range(a, 6), count_in_range(times, 3, 8))

[3, 4] [-1, -1] 6


In [ ]:
import random

def k_closest(a, k, x):
    lo, hi = 0, len(a) - k             # candidate window starts: 0 .. n-k
    while lo < hi:
        mid = (lo + hi) // 2
        if x - a[mid] > a[mid + k] - x:    # left edge is farther than the item just outside
            lo = mid + 1                   # move the window right
        else:
            hi = mid                       # keep mid: ties favour smaller values
    return a[lo:lo + k]

def brute(a, k, x):
    return sorted(sorted(a, key=lambda v: (abs(v - x), v))[:k])

assert k_closest([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert k_closest([1, 2, 3, 4, 5], 4, 10) == [2, 3, 4, 5]
random.seed(14)
for _ in range(300):
    arr = sorted(random.randint(0, 20) for _ in range(random.randint(1, 10)))
    k, x = random.randint(1, len(arr)), random.randint(-3, 24)
    assert k_closest(arr, k, x) == brute(arr, k, x)
print(k_closest([1, 2, 3, 4, 5], 4, 3), k_closest([1, 1, 2, 6, 9], 2, 5))

[1, 2, 3, 4] [2, 6]


## 4.8 Binary Search on the Answer

In [ ]:
def smallest_feasible(lo, hi, feasible):
    """Smallest x in [lo, hi] with feasible(x) True (False...False True...True)."""
    while lo < hi:
        mid = (lo + hi) // 2           # rounds down
        if feasible(mid):
            hi = mid                   # mid works; maybe a smaller one does too
        else:
            lo = mid + 1
    return lo

def largest_feasible(lo, hi, feasible):
    """Largest x in [lo, hi] with feasible(x) True (True...True False...False)."""
    while lo < hi:
        mid = (lo + hi + 1) // 2       # rounds UP: needed because we set lo = mid
        if feasible(mid):
            lo = mid                   # mid works; maybe a larger one does too
        else:
            hi = mid - 1
    return lo

print(smallest_feasible(0, 100, lambda x: x * x >= 50))
print(largest_feasible(0, 100, lambda x: x * x <= 50))

8
7


In [ ]:
import math

def int_sqrt(n):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2       # round up because we assign lo = mid
        if mid * mid <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo

assert int_sqrt(8) == 2 and int_sqrt(0) == 0 and int_sqrt(1) == 1
assert all(int_sqrt(n) == math.isqrt(n) for n in range(2000))
assert int_sqrt(10 ** 18 + 5) == math.isqrt(10 ** 18 + 5)
print(int_sqrt(8), int_sqrt(99), int_sqrt(10 ** 18))

2 9 1000000000


In [ ]:
def min_eating_speed(piles, h):
    def hours(k):
        return sum(-(-p // k) for p in piles)      # -(-p // k) = ceil(p / k) in integers
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(mid) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo

assert min_eating_speed([3, 6, 7, 11], 8) == 4
assert min_eating_speed([30, 11, 23, 4, 20], 5) == 30
assert min_eating_speed([30, 11, 23, 4, 20], 6) == 23
print(min_eating_speed([3, 6, 7, 11], 8), min_eating_speed([30, 11, 23, 4, 20], 6))

4 23


In [ ]:
def ship_within_days(weights, days):
    def days_needed(cap):
        d, load = 1, 0
        for w in weights:
            if load + w > cap:         # does not fit today: next day
                d, load = d + 1, 0
            load += w
        return d
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if days_needed(mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo

assert ship_within_days(list(range(1, 11)), 5) == 15
assert ship_within_days([3, 2, 2, 4, 1, 4], 3) == 6
assert ship_within_days([1, 2, 3, 1, 1], 4) == 3
print(ship_within_days(list(range(1, 11)), 5), ship_within_days([3, 2, 2, 4, 1, 4], 3))

15 6


In [ ]:
def max_min_distance(stalls, cows):
    s = sorted(stalls)
    def feasible(d):                   # can `cows` cows sit at least d apart?
        placed, last = 1, s[0]
        for x in s[1:]:
            if x - last >= d:
                placed, last = placed + 1, x
        return placed >= cows
    lo, hi = 1, s[-1] - s[0]           # d = 1 always works for distinct stalls
    while lo < hi:
        mid = (lo + hi + 1) // 2       # round UP: the true branch sets lo = mid
        if feasible(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

assert max_min_distance([1, 2, 4, 8, 9], 3) == 3
assert max_min_distance([1, 2, 3, 4, 7], 3) == 3
assert max_min_distance([5, 1, 9], 2) == 8 and max_min_distance([1, 2, 4, 8, 9], 5) == 1
print(max_min_distance([1, 2, 4, 8, 9], 3), max_min_distance([1, 2, 3, 4, 7], 3))

3 3


## 4.9 Rotated Arrays, Peaks and 2D Matrices

These look like they break the "sorted" rule, but each hides an order that lets you discard half. Always ask: *after looking at mid, which half cannot hold the answer, and how do I know?*

In [ ]:
def find_min_rotated(a):
    lo, hi = 0, len(a) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] > a[hi]:             # mid sits in the left (bigger) run
            lo = mid + 1
        else:                          # mid is in the right run: it may be the minimum
            hi = mid
    return a[lo]

assert find_min_rotated([4, 5, 6, 7, 0, 1, 2]) == 0
assert find_min_rotated([1, 2, 3]) == 1 and find_min_rotated([9]) == 9 and find_min_rotated([2, 1]) == 1
print(find_min_rotated([4, 5, 6, 7, 0, 1, 2]), find_min_rotated([11, 13, 15, 17]))

0 11


In [ ]:
import random

def search_rotated(a, target):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == target:
            return mid
        if a[lo] <= a[mid]:                      # left half a[lo..mid] is sorted
            if a[lo] <= target < a[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                    # right half a[mid..hi] is sorted
            if a[mid] < target <= a[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

assert search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1 and search_rotated([3, 1], 1) == 1
random.seed(22)
for _ in range(100):
    base = sorted(random.sample(range(40), random.randint(1, 10)))
    r = random.randint(0, len(base) - 1)
    arr = base[r:] + base[:r]
    for t in range(-1, 42):
        assert search_rotated(arr, t) == (arr.index(t) if t in arr else -1)
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0), search_rotated([4, 5, 6, 7, 0, 1, 2], 3))

4 -1


In [ ]:
def find_peak(a):
    lo, hi = 0, len(a) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < a[mid + 1]:        # rising: a peak is to the right
            lo = mid + 1
        else:                          # falling: mid or something left of it is a peak
            hi = mid
    return lo

def is_peak(a, i):                     # outside the array counts as minus infinity
    left = a[i - 1] if i > 0 else float("-inf")
    right = a[i + 1] if i < len(a) - 1 else float("-inf")
    return a[i] > left and a[i] > right

assert find_peak([1, 2, 1, 3, 5, 6, 4]) == 5
assert find_peak([7]) == 0 and find_peak([1, 2, 3]) == 2 and find_peak([3, 2, 1]) == 0
for arr in ([1, 2, 1, 3, 5, 6, 4], [5, 1, 4, 2, 6], [1, 3, 2, 4, 3, 5]):
    assert is_peak(arr, find_peak(arr))
print(find_peak([1, 2, 1, 3, 5, 6, 4]), find_peak([1, 2, 3]))

5 2


In [ ]:
def search_matrix(m, target):
    if not m or not m[0]:
        return False
    rows, cols = len(m), len(m[0])
    lo, hi = 0, rows * cols - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r, c = divmod(mid, cols)       # flat index -> (row, column)
        if m[r][c] == target:
            return True
        if m[r][c] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

grid = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert search_matrix(grid, 3) is True and search_matrix(grid, 13) is False
assert all(search_matrix(grid, v) for row in grid for v in row)
assert search_matrix([], 1) is False and search_matrix([[]], 1) is False
print(search_matrix(grid, 3), search_matrix(grid, 13))

True False


## Chapter recap

## Practice set

Try each problem on paper first. Problems are graded from easy to hard. "Predict" problems show code without output: write your guess, then read the answer.

In [ ]:
words = ["b", "a", "C"]
print(sorted(words))
print(sorted(words, key=str.lower))
print(sorted(words, reverse=True))

In [ ]:
import bisect
a = [1, 2, 2, 3]
print(bisect.bisect_left(a, 2), bisect.bisect_right(a, 2), bisect.bisect_left(a, 4), bisect.bisect_right(a, 0))

In [ ]:
def bs(a, t):
    lo, hi = 0, len(a) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] == t:
            return mid
        elif a[mid] < t:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print(bs([1, 3, 5], 5), bs([1, 3, 5], 1), bs([5], 5))

In [ ]:
def merge_bad(a, b):
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    return out

print(merge_bad([1, 2, 9], [3]))

In [ ]:
def merge_bad(intervals):
    result = []
    for s, e in sorted(intervals):
        if result and s <= result[-1][1]:
            result[-1][1] = e
        else:
            result.append([s, e])
    return result

print(merge_bad([[1, 10], [2, 3]]))

In [ ]:
scores = [70, 95, 82]
ranked = scores.sort(reverse=True)
print(ranked, scores)

In [ ]:
print(sorted([("b", 2), ("a", 2), ("c", 1)], key=lambda x: x[1]))

In [ ]:
def partition(a, lo, hi):
    pivot, i = a[hi], lo - 1
    for j in range(lo, hi):
        if a[j] <= pivot:
            i += 1
            a[i], a[j] = a[j], a[i]
    a[i + 1], a[hi] = a[hi], a[i + 1]
    return i + 1

arr = [3, 8, 2, 5, 1, 4, 7, 6]
print(partition(arr, 0, 7), arr)

In [ ]:
lo, hi = 5, 6
print((lo + hi) // 2, (lo + hi + 1) // 2)

In [ ]:
def sort_colors_bad(a):
    a = a[:]
    lo, mid, hi = 0, 0, len(a) - 1
    while mid <= hi:
        if a[mid] == 0:
            a[lo], a[mid] = a[mid], a[lo]
            lo += 1; mid += 1
        elif a[mid] == 1:
            mid += 1
        else:
            a[mid], a[hi] = a[hi], a[mid]
            hi -= 1; mid += 1
    return a

print(sort_colors_bad([1, 2, 0]))

In [ ]:
import bisect

letters = ["c", "f", "j"]
def next_letter(target):
    return letters[bisect.bisect_right(letters, target) % len(letters)]

assert [next_letter(t) for t in ("a", "c", "j")] == ["c", "f", "c"]
print([next_letter(t) for t in ("a", "c", "j")])

['c', 'f', 'c']


In [ ]:
def kth_smallest(matrix, k):
    n = len(matrix)
    def count_at_most(x):              # staircase walk from the bottom-left corner
        r, c, total = n - 1, 0, 0
        while r >= 0 and c < n:
            if matrix[r][c] <= x:
                total += r + 1         # the whole column above is also <= x
                c += 1
            else:
                r -= 1
        return total
    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo < hi:                     # smallest value whose count reaches k
        mid = (lo + hi) // 2
        if count_at_most(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

grid = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
assert kth_smallest(grid, 8) == 13
assert kth_smallest(grid, 1) == 1 and kth_smallest(grid, 9) == 15
print(kth_smallest(grid, 8))

13


In [ ]:
nums = [3, 30, 34, 5, 9]
print("".join(sorted(map(str, nums), reverse=True)))

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch04.html